# Qwen BPMN Worker — procédure et narration

Ce notebook Kaggle charge **Qwen une seule fois**, puis exécute les
générateurs versionnés du projet.

Modes disponibles dans la cellule de configuration :

- `procedure` : génère uniquement la procédure ;
- `narrative` : génère uniquement la description narrative ;
- `both` : génère les deux indépendamment avec le même modèle chargé.

Les deux chaînes restent séparées :

```text
operation_contexts.json
    ├── procedure_generation
    │      └── generated_procedure.json
    │
    └── narrative_generation
           + narrative_plan.json
           └── generated_narrative.json
```

Activez **GPU** et **Internet** dans les paramètres Kaggle.


In [1]:
# Install a P100-compatible CUDA stack.
# CUDA 12.6 supports both P100 and newer Kaggle GPUs.

import subprocess
import sys


def pip_install(*arguments: str) -> None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            *arguments,
        ],
        check=True,
    )


pip_install(
    "--force-reinstall",
    "--index-url",
    "https://download.pytorch.org/whl/cu126",
    "torch==2.10.0",
)

pip_install(
    "transformers==4.57.6",
    "accelerate>=1.0",
    "huggingface_hub>=0.30",
    "pydantic>=2.7,<3",
)

pip_install(
    "--force-reinstall",
    "--no-deps",
    "bitsandbytes==0.49.0",
)

print("P100-compatible dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 44.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00


In [ ]:
# Static defaults. Automated runs override these values from run_manifest.json.

from pathlib import Path

MANUAL_RUN_MODE = "both"
REPO_URL = (
    "https://github.com/nour0205/"
    "bpmn-procedure-generator.git"
)
DEFAULT_REPO_BRANCH = "main"
DEFAULT_MODEL_NAME = "Qwen/Qwen3-8B"

PROCEDURE_PROMPT_VERSION = "independent-procedure-v1.0"
NARRATIVE_PROMPT_VERSION = "independent-narrative-v1.2"

INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "bpmn-procedure-generator"
OUTPUT_DIR = WORKING_DIR

MAX_ATTEMPTS_PER_OPERATION = 3
MAX_NEW_TOKENS_PER_OPERATION = 420
MAX_ATTEMPTS_PER_UNIT = 3
MAX_NEW_TOKENS_PER_UNIT = 650


In [ ]:
# Detect the prepared input package before cloning the project.

import json
from datetime import datetime, timezone


def find_unique_input(filename: str, *, required: bool = True):
    matches = sorted(INPUT_ROOT.rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(
                f"{filename} was not found under /kaggle/input."
            )
        return None
    if len(matches) > 1:
        formatted = "\n".join(f"- {path}" for path in matches)
        raise RuntimeError(
            f"Several {filename} files were detected:\n"
            f"{formatted}\nAttach only one process input dataset."
        )
    return matches[0]


RUN_MANIFEST_PATH = find_unique_input(
    "run_manifest.json",
    required=False,
)

if RUN_MANIFEST_PATH is not None:
    run_manifest = json.loads(
        RUN_MANIFEST_PATH.read_text(encoding="utf-8")
    )
else:
    run_manifest = {
        "schema_version": "1.0",
        "run_id": (
            "manual-"
            + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        ),
        "process_slug": "manual",
        "generation_mode": MANUAL_RUN_MODE,
        "model_name": DEFAULT_MODEL_NAME,
        "git_commit": None,
        "git_branch": DEFAULT_REPO_BRANCH,
        "git_dirty": False,
    }

RUN_MODE = run_manifest.get("generation_mode", MANUAL_RUN_MODE)
MODEL_NAME = run_manifest.get("model_name", DEFAULT_MODEL_NAME)
REPO_BRANCH = run_manifest.get("git_branch") or DEFAULT_REPO_BRANCH
PROCESS_TITLE_OVERRIDE = run_manifest.get("process_title")

allowed_modes = {"procedure", "narrative", "both"}
if RUN_MODE not in allowed_modes:
    raise ValueError(
        "generation_mode must be procedure, narrative or both."
    )

OPERATION_CONTEXTS_PATH = find_unique_input(
    "operation_contexts.json"
)
input_payload = json.loads(
    OPERATION_CONTEXTS_PATH.read_text(encoding="utf-8")
)

required_context_fields = {
    "process_id",
    "operation_count",
    "contexts",
}
missing_fields = required_context_fields - set(input_payload)
if missing_fields:
    raise ValueError(
        "operation_contexts.json is missing: "
        + ", ".join(sorted(missing_fields))
    )

expected_process_id = run_manifest.get("process_id")
if (
    expected_process_id
    and expected_process_id != input_payload["process_id"]
):
    raise ValueError(
        "run_manifest.json and operation_contexts.json use "
        "different process IDs."
    )

NARRATIVE_PLAN_PATH = None
if RUN_MODE in {"narrative", "both"}:
    NARRATIVE_PLAN_PATH = find_unique_input("narrative_plan.json")
    narrative_plan_payload = json.loads(
        NARRATIVE_PLAN_PATH.read_text(encoding="utf-8")
    )
    if (
        narrative_plan_payload.get("process_id")
        != input_payload["process_id"]
    ):
        raise ValueError(
            "narrative_plan.json and operation_contexts.json use "
            "different process IDs."
        )

PROCESS_TITLE = (
    PROCESS_TITLE_OVERRIDE
    or input_payload.get("title")
    or input_payload["process_id"]
)

print("Run ID:", run_manifest["run_id"])
print("Run mode:", RUN_MODE)
print("Process:", PROCESS_TITLE)
print("Process ID:", input_payload["process_id"])
print("Git branch:", REPO_BRANCH)
print("Expected Git commit:", run_manifest.get("git_commit"))
print("Model:", MODEL_NAME)


In [ ]:
# Load the exact version-controlled project source.

import base64
import os
import shutil
import subprocess
import sys


def find_attached_project_src(root: Path):
    candidates = []
    for marker in root.rglob(
        "src/procedure_generation/__init__.py"
    ):
        src_dir = marker.parents[1]
        if (
            src_dir
            / "narrative_generation"
            / "__init__.py"
        ).exists():
            candidates.append(src_dir.resolve())

    candidates = sorted(set(candidates))
    if len(candidates) > 1:
        formatted = "\n".join(f"- {item}" for item in candidates)
        raise RuntimeError(
            "Several attached project copies were detected:\n"
            f"{formatted}\nKeep only one project dataset."
        )
    return candidates[0] if candidates else None


def git_commit_for(path: Path):
    try:
        completed = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=path,
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception:
        return None
    return completed.stdout.strip() or None


project_src = find_attached_project_src(INPUT_ROOT)

if project_src is None:
    github_token = None
    try:
        from kaggle_secrets import UserSecretsClient
        github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        github_token = os.environ.get("GITHUB_TOKEN")

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    clone_command = [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        str(REPO_DIR),
    ]

    if github_token:
        basic_token = base64.b64encode(
            ("x-access-token:" + github_token).encode("utf-8")
        ).decode("ascii")
        clone_command = [
            "git",
            "-c",
            (
                "http.extraHeader=AUTHORIZATION: basic "
                + basic_token
            ),
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ]

    subprocess.run(clone_command, check=True)
    project_src = REPO_DIR / "src"

required_packages = [
    project_src / "procedure_generation" / "__init__.py",
    project_src / "narrative_generation" / "__init__.py",
]
missing_packages = [path for path in required_packages if not path.exists()]
if missing_packages:
    formatted = "\n".join(f"- {path}" for path in missing_packages)
    raise FileNotFoundError(
        "The project source does not contain the required packages:\n"
        + formatted
    )

PROJECT_GIT_COMMIT = git_commit_for(project_src.parent)
EXPECTED_GIT_COMMIT = run_manifest.get("git_commit")
if (
    EXPECTED_GIT_COMMIT
    and PROJECT_GIT_COMMIT
    and EXPECTED_GIT_COMMIT != PROJECT_GIT_COMMIT
):
    raise RuntimeError(
        "The remote repository commit does not match the prepared run. "
        f"Expected {EXPECTED_GIT_COMMIT}, cloned {PROJECT_GIT_COMMIT}. "
        "Commit and push the local changes before running Kaggle."
    )

sys.path.insert(0, str(project_src))

from procedure_generation import (
    ProcedureGenerationConfig,
    run_procedure_generation,
)
from procedure_generation.model_adapter import (
    QwenTextGenerator as ProcedureQwenTextGenerator,
)
from narrative_generation import (
    NarrativeGenerationConfig,
    run_narrative_generation,
)
from narrative_generation.model_adapter import (
    QwenTextGenerator as NarrativeQwenTextGenerator,
)

print("Project source:", project_src)
print("Cloned Git commit:", PROJECT_GIT_COMMIT)


In [5]:
# Load Qwen once in NF4 4-bit mode

import gc
import platform

import torch
import transformers
from huggingface_hub import login
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable a GPU "
        "accelerator in Kaggle settings."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
supported_architectures = torch.cuda.get_arch_list()

print("GPU:", gpu_name)
print("GPU capability:", gpu_capability)
print(
    "PyTorch CUDA:",
    torch.version.cuda,
)
print(
    "Supported architectures:",
    supported_architectures,
)

required_architecture = (
    f"sm_{gpu_capability[0]}"
    f"{gpu_capability[1]}"
)

if required_architecture not in supported_architectures:
    raise RuntimeError(
        "The installed PyTorch build does not support "
        f"{gpu_name} ({required_architecture}). "
        f"Supported architectures: "
        f"{supported_architectures}"
    )

hf_token = None

try:
    from kaggle_secrets import (
        UserSecretsClient,
    )

    hf_token = (
        UserSecretsClient()
        .get_secret("HF_TOKEN")
    )
except Exception:
    hf_token = os.environ.get(
        "HF_TOKEN"
    )

if hf_token:
    login(token=hf_token)
    print(
        "Hugging Face authentication "
        "successful."
    )
else:
    print(
        "No HF_TOKEN found. Loading "
        "continues if the model is public."
    )

os.environ[
    "HF_HUB_DOWNLOAD_TIMEOUT"
] = "600"
os.environ[
    "HF_HUB_ETAG_TIMEOUT"
] = "60"

print(
    "Python:",
    platform.python_version(),
)
print(
    "PyTorch:",
    torch.__version__,
)
print(
    "Transformers:",
    transformers.__version__,
)
print(
    "GPU:",
    torch.cuda.get_device_name(0),
)

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization_config = (
    BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=(
            compute_dtype
        ),
        bnb_4bit_use_double_quant=True,
    )
)

tokenizer = (
    AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=hf_token,
        use_fast=True,
    )
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = (
        tokenizer.eos_token
    )

model = (
    AutoModelForCausalLM
    .from_pretrained(
        MODEL_NAME,
        token=hf_token,
        quantization_config=(
            quantization_config
        ),
        device_map="auto",
        dtype=compute_dtype,
        low_cpu_mem_usage=True,
    )
)

model.eval()

print("Model loaded:", MODEL_NAME)
print(
    "Device map:",
    model.hf_device_map,
)

gc.collect()
torch.cuda.empty_cache()


No HF_TOKEN found. Loading continues if the model is public.
Python: 3.12.13
PyTorch: 2.10.0+cu128
Transformers: 4.57.6
GPU: Tesla T4


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.19G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-8B
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.layers.32': 1, 'model.layers.33': 1, 'model.layers.34': 1, 'model.layers.35': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


In [ ]:
# Run procedure generation independently.

procedure_result = None
procedure_error = None

if RUN_MODE in {"procedure", "both"}:
    try:
        procedure_text_generator = ProcedureQwenTextGenerator(
            model=model,
            tokenizer=tokenizer,
        )
        procedure_config = ProcedureGenerationConfig(
            model_name=MODEL_NAME,
            prompt_version=PROCEDURE_PROMPT_VERSION,
            max_attempts_per_operation=MAX_ATTEMPTS_PER_OPERATION,
            max_new_tokens_per_operation=(
                MAX_NEW_TOKENS_PER_OPERATION
            ),
            use_model=True,
            cache_dir=OUTPUT_DIR / "procedure_cache",
            title_override=PROCESS_TITLE,
        )
        procedure_result = run_procedure_generation(
            operation_contexts_path=OPERATION_CONTEXTS_PATH,
            output_dir=OUTPUT_DIR,
            text_generator=procedure_text_generator,
            config=procedure_config,
        )
        print(
            "Generated procedure:",
            procedure_result.generated_procedure_path,
        )
    except Exception as exc:
        procedure_error = {
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }
        print("Procedure generation failed:", procedure_error)
    finally:
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("Procedure generation skipped.")


In [ ]:
# Run narrative generation independently.

narrative_result = None
narrative_error = None

if RUN_MODE in {"narrative", "both"}:
    try:
        narrative_text_generator = NarrativeQwenTextGenerator(
            model=model,
            tokenizer=tokenizer,
        )
        narrative_config = NarrativeGenerationConfig(
            model_name=MODEL_NAME,
            prompt_version=NARRATIVE_PROMPT_VERSION,
            max_attempts_per_unit=MAX_ATTEMPTS_PER_UNIT,
            max_new_tokens_per_unit=MAX_NEW_TOKENS_PER_UNIT,
            use_model=True,
            cache_dir=OUTPUT_DIR / "narrative_cache",
        )
        narrative_result = run_narrative_generation(
            narrative_plan_path=NARRATIVE_PLAN_PATH,
            operation_contexts_path=OPERATION_CONTEXTS_PATH,
            output_dir=OUTPUT_DIR,
            text_generator=narrative_text_generator,
            config=narrative_config,
        )
        print(
            "Generated narrative:",
            narrative_result.generated_narrative_path,
        )
    except Exception as exc:
        narrative_error = {
            "error_type": type(exc).__name__,
            "error_message": str(exc),
        }
        print("Narrative generation failed:", narrative_error)
    finally:
        gc.collect()
        torch.cuda.empty_cache()
else:
    print("Narrative generation skipped.")


In [ ]:
# Write one run-identified result while preserving independent statuses.


def component_result(
    result,
    error,
    *,
    generated_attr,
    validation_attr,
    preview_attr,
):
    if error is not None:
        return {"status": "failed", **error}
    if result is None:
        return {"status": "skipped"}
    return {
        "status": "success",
        "generated_file": getattr(result, generated_attr).name,
        "validation_file": getattr(result, validation_attr).name,
        "preview_file": getattr(result, preview_attr).name,
        "quality": result.validation_report,
    }


run_result = {
    "schema_version": "1.0",
    "run_id": run_manifest["run_id"],
    "generation_mode": RUN_MODE,
    "process_slug": run_manifest.get("process_slug", "manual"),
    "process_id": input_payload["process_id"],
    "process_title": PROCESS_TITLE,
    "model_name": MODEL_NAME,
    "git_commit": PROJECT_GIT_COMMIT,
    "procedure": component_result(
        procedure_result,
        procedure_error,
        generated_attr="generated_procedure_path",
        validation_attr="validation_report_path",
        preview_attr="preview_path",
    ),
    "narrative": component_result(
        narrative_result,
        narrative_error,
        generated_attr="generated_narrative_path",
        validation_attr="validation_report_path",
        preview_attr="preview_path",
    ),
}

run_result_path = OUTPUT_DIR / "run_result.json"
run_result_path.write_text(
    json.dumps(run_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

print(json.dumps(run_result, ensure_ascii=False, indent=2))
print("Run result:", run_result_path)

for filename in run_manifest.get("expected_outputs", []):
    path = OUTPUT_DIR / filename
    print(("OK" if path.exists() else "MISSING"), path)
